# Naive Bayes for the TREC06 Corpus

This notebook is my implementation of a Naive Bayes spam classifier for the TREC06 email dataset. The dataset has a `labels` file and folders of email messages, so most of the work is reading those files, splitting them into train/test sets, and checking how well the classifier works.

For this notebook I first used the raw email files, then tested smoothing and the 200-word version. At the end I also tried a cleaned-body version just to see if it would help.

The first model has no smoothing, so it is basically using $\lambda = 0$.

# 2.1 Classifier Construction and Evaluation

## Load the Labels
I read the labels file here and turn each line into a label plus the path to the email. I also keep track of missing files so they do not get included later.

In [1]:
# Load the dataset labels and email paths
from pathlib import Path

dataset_dir = Path("../trec06p-cs280")
label_file = dataset_dir / "labels"

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)


def load_labels(label_file):
    valid_records = []
    missing_records = []

    with label_file.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            label, relative_path = line.split(maxsplit=1)

            email_path = dataset_dir / (
                Path(relative_path.removeprefix("../")).with_suffix(".eml")
            )

            record = {
                "label": label,
                "relative_path": relative_path,
                "path": email_path,
            }

            # Only use files that are actually in the dataset folder
            if email_path.exists():
                valid_records.append(record)
            else:
                missing_records.append(record)

    return valid_records, missing_records

valid_records, missing_records = load_labels(label_file)

print("Total label records:", len(valid_records) + len(missing_records))
print("Valid records:", len(valid_records))
print("Missing records:", len(missing_records))

Total label records: 37822
Valid records: 37782
Missing records: 40


## Train and Test Split
I split spam and ham separately before joining them back together. This keeps both classes represented in the 70/30 split. The seed is fixed so I can reproduce the same split.

In [2]:
# 70/30 splitting
import random

def split_dataset(records, train_ratio=0.7, seed=42):
    random_gen = random.Random(seed)

    spam_records = [
        record for record in records
        if record["label"] == "spam"
    ]

    ham_records = [
        record for record in records
        if record["label"] == "ham"
    ]

    random_gen.shuffle(spam_records)
    random_gen.shuffle(ham_records)

    spam_split = int(len(spam_records)*train_ratio)
    ham_split = int(len(ham_records)*train_ratio)

    train_data = (
        spam_records[:spam_split] +
        ham_records[:ham_split]
    )

    test_data = (
        spam_records[spam_split:] +
        ham_records[ham_split:]
    )

    return train_data, test_data

train_data, test_data = split_dataset(valid_records, train_ratio=0.7, seed=42)
print(len(valid_records))
print(len(train_data))
print(len(test_data))

37782
26447
11335


In [3]:
# Check how many spam and ham emails ended up in each set
from collections import Counter

for name, data in (("TRAIN", train_data), ("TEST", test_data)):
    label_counts = Counter(record["label"] for record in data)

    print(name)
    print("Spam:", label_counts["spam"])
    print("Ham:", label_counts["ham"])
    print()

TRAIN
Spam: 17417
Ham: 9030

TEST
Spam: 7465
Ham: 3870



In [4]:
# Make sure the same email is not in both sets
train_paths = {
    record["path"]
    for record in train_data
}

test_paths = {
    record["path"]
    for record in test_data
}

overlap = train_paths & test_paths

print("Overlapping emails:", len(overlap))

assert len(overlap) == 0

Overlapping emails: 0


In [5]:
# Save the train and test lists using the same format as labels
def save_records(records, path):
    with path.open("w") as f:
        for record in records:
            f.write(f"{record['label']} {record['relative_path']}\n")

    print("Saved:", path)

save_records(train_data, results_dir / "train_set.txt")
save_records(test_data, results_dir / "test_set.txt")

Saved: ../results/train_set.txt
Saved: ../results/test_set.txt


## Tokenizing the Emails
For the words, I only kept alphabetic strings that match the rule in the instructions. The vocabulary is just the unique words found in the training emails.

In [6]:
# Token rule for words in the emails
import re

# Letters with whitespace before and whitespace, comma, or period after.
# The start and end of the text are handled too.
word_pattern = re.compile(r"(?<!\S)[a-zA-Z]+(?=[\s,.]|\Z)")

def tokenize(text):
    return word_pattern.findall(text)

print(tokenize("Hello, world. Visit www.site.com or e-mail me@site.com for FREE money!"))

['Hello', 'world', 'Visit', 'www', 'or', 'for', 'FREE']


In [7]:
# Build the vocabulary and count how many documents contain each word
def read_email(path):
    with path.open(
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:
        return f.read()

def get_words(text):
    # I only need whether a word appears in the email, so I use a set
    return {word.lower() for word in tokenize(text)}

def email_words(path):
    return get_words(read_email(path))

def count_documents(records, read_words):
    spam_df = Counter()   # spam document count for each word
    ham_df = Counter()    # ham document count for each word
    spam_doc = 0
    ham_doc = 0

    for record in records:
        words = read_words(record["path"])

        if record["label"] == "spam":
            spam_doc += 1
            spam_df.update(words)

        elif record["label"] == "ham":
            ham_doc += 1
            ham_df.update(words)

    return spam_df, ham_df, spam_doc, ham_doc

train_stats = count_documents(train_data, email_words)
spam_df, ham_df, spam_doc, ham_doc = train_stats

vocabs = set(spam_df) | set(ham_df)

In [8]:
# Quick look at the vocabulary and counts
print("Vocabulary size:", len(vocabs))
print("Words found only in spam:", len(vocabs - set(ham_df)))
print("Words found only in ham:", len(vocabs - set(spam_df)))

print("Spam training documents:", spam_doc)
print("Ham training documents:", ham_doc)

print()
print("Words in the most spam documents:", spam_df.most_common(10))
print("Words in the most ham documents:", ham_df.most_common(10))

Vocabulary size: 78278
Words found only in spam: 15829
Words found only in ham: 46814
Spam training documents: 17417
Ham training documents: 9030

Words in the most spam documents: [('by', 17417), ('from', 17417), ('with', 17417), ('id', 17393), ('for', 16495), ('esmtp', 14540), ('a', 11079), ('in', 10651), ('is', 10313), ('to', 9903)]
Words in the most ham documents: [('by', 9030), ('from', 9020), ('with', 9008), ('id', 8999), ('for', 8860), ('the', 8312), ('to', 7951), ('esmtp', 7552), ('a', 7536), ('and', 7141)]


## Training the First Model
Here I compute the class priors and the word probabilities for the first version, without smoothing.


In [9]:
# Priors for spam and ham
import math

total_train = spam_doc + ham_doc

prior_spam = spam_doc / total_train
prior_ham = ham_doc / total_train

print("Training documents:", total_train)

print("\nPrior probabilities")
print("P(spam):", prior_spam)
print("P(ham):", prior_ham)

Training documents: 26447

Prior probabilities
P(spam): 0.658562407834537
P(ham): 0.341437592165463


In [10]:
# Store likelihoods as logs. If a word never appears in a class,
# its probability is 0, so the log value is -inf.
def train(stats, vocab):
    spam_df, ham_df, spam_doc, ham_doc = stats

    log_spam = {}
    log_ham = {}

    for word in vocab:
        log_spam[word] = math.log(spam_df[word] / spam_doc) if spam_df[word] else -math.inf
        log_ham[word] = math.log(ham_df[word] / ham_doc) if ham_df[word] else -math.inf

    return log_spam, log_ham

baseline_model = train(train_stats, vocabs)

## Classification
I used log probabilities so I can add scores instead of multiplying many small numbers. The class with the higher score becomes the prediction.

In [11]:
# Use logs so the scores do not get too tiny
def score(words, model):
    log_spam, log_ham = model

    spam_score = math.log(prior_spam)
    ham_score = math.log(prior_ham)

    for word in words:
        if word in log_spam:
            spam_score += log_spam[word]
            ham_score += log_ham[word]

    return spam_score, ham_score

def classify(words, model):
    spam_score, ham_score = score(words, model)

    if spam_score > ham_score:
        return "spam"
    else:
        return "ham"

def spam_posterior(spam_score, ham_score):
    # Convert the two log scores back into a spam probability
    if spam_score == -math.inf and ham_score == -math.inf:
        return None   # both likelihoods are 0, so the posterior is 0/0

    biggest = max(spam_score, ham_score)
    spam = math.exp(spam_score - biggest)
    ham = math.exp(ham_score - biggest)

    return spam / (spam + ham)

In [12]:
# Try a small made-up message
unknown_message = """
Congratulations, you won free money. Claim your prize now.
"""

def classify_unknown(email_text, model):
    words = get_words(email_text)
    spam_score, ham_score = score(words, model)

    print("Prediction:", classify(words, model))
    print("Spam score:", spam_score)
    print("Ham score:", ham_score)
    print("P(spam | message):", spam_posterior(spam_score, ham_score))
    print("Words used:", sorted(word for word in words if word in model[0]))

classify_unknown(unknown_message, baseline_model)

Prediction: spam
Spam score: -28.378194428334684
Ham score: -33.933756837204335
P(spam | message): 0.996148993934677
Words used: ['claim', 'congratulations', 'free', 'money', 'now', 'prize', 'won', 'you', 'your']


In [13]:
# Read the test set once so the later tests can reuse it
test_docs = [
    (record["label"], email_words(record["path"]))
    for record in test_data
]

# Quick check on the first 10 test emails
for actual, words in test_docs[:10]:
    print("Predicted:", classify(words, baseline_model), " Actual:", actual)

Predicted: spam  Actual: spam
Predicted: spam  Actual: spam
Predicted: spam  Actual: spam
Predicted: spam  Actual: spam
Predicted: spam  Actual: spam
Predicted: spam  Actual: spam
Predicted: spam  Actual: spam
Predicted: spam  Actual: spam
Predicted: ham  Actual: spam
Predicted: spam  Actual: spam


## Evaluation
These functions count TP, TN, FP, and FN, then compute precision and recall. I also included F1 and accuracy so it is easier to compare runs.

In [14]:
def precision_recall(TP, FP, FN):
    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0

    return precision, recall

def evaluate(docs, model):
    TP = TN = FP = FN = 0

    for actual, words in docs:
        predicted = classify(words, model)

        if actual == "spam" and predicted == "spam":
            TP += 1
        elif actual == "ham" and predicted == "ham":
            TN += 1
        elif actual == "ham" and predicted == "spam":
            FP += 1
        elif actual == "spam" and predicted == "ham":
            FN += 1

    precision, recall = precision_recall(TP, FP, FN)
    f1_score = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = (TP + TN) / len(docs)

    return {
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "accuracy": accuracy,
    }

In [15]:
# Test the no-smoothing model
baseline_result = evaluate(test_docs, baseline_model)

for metric, value in baseline_result.items():
    print(metric, ":", value)

# With no smoothing, unseen words can make a class score become -inf.
# I counted how often both class scores end up like that.
both_zero = sum(
    1 for actual, words in test_docs
    if score(words, baseline_model) == (-math.inf, -math.inf)
)

print()
print("Test emails with zero probability for both classes:", both_zero, "of", len(test_docs))

TP : 6851
TN : 3866
FP : 4
FN : 614
precision : 0.999416484318016
recall : 0.9177494976557268
f1_score : 0.9568435754189945
accuracy : 0.9454786060873401

Test emails with zero probability for both classes: 632 of 11335


# 2.2 Lambda Smoothing

## Lambda Smoothing
Next I tried the lambda values from the instructions: `2.0`, `1.0`, `0.5`, `0.1`, and `0.005`.

In [16]:
# Function for smoothed probabilities
def smoothed_likelihood(doc_count, class_docs, vocab_size, lambda_value):
    return (doc_count + lambda_value) / (class_docs + lambda_value * vocab_size)

def train_smoothed(stats, vocab, lambda_value):
    spam_df, ham_df, spam_doc, ham_doc = stats
    vocab_size = len(vocab)

    log_spam = {}
    log_ham = {}

    for word in vocab:
        log_spam[word] = math.log(smoothed_likelihood(spam_df[word], spam_doc, vocab_size, lambda_value))
        log_ham[word] = math.log(smoothed_likelihood(ham_df[word], ham_doc, vocab_size, lambda_value))

    return log_spam, log_ham

In [17]:
# Run all required lambda values and store the results
lambda_values = [2.0, 1.0, 0.5, 0.1, 0.005]

results = []

for lambda_value in lambda_values:
    model = train_smoothed(train_stats, vocabs, lambda_value)

    results.append({"lambda": lambda_value, **evaluate(test_docs, model)})

In [18]:
# Print the results as a small table
import csv

def print_table(rows, columns):
    lines = []

    for row in rows:
        lines.append([
            str(round(row[column], 6)) if isinstance(row[column], float) else str(row[column])
            for column in columns
        ])

    widths = [
        max(len(column), max(len(line[index]) for line in lines))
        for index, column in enumerate(columns)
    ]

    print(" | ".join(column.ljust(width) for column, width in zip(columns, widths)))
    print("-+-".join("-" * width for width in widths))

    for line in lines:
        print(" | ".join(value.ljust(width) for value, width in zip(line, widths)))

def save_csv(path, rows):
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

    print("Saved:", path)

metric_columns = ["lambda", "TP", "TN", "FP", "FN", "precision", "recall", "f1_score", "accuracy"]

print_table(results, metric_columns)
print()
save_csv(results_dir / "lambda_metrics.csv", results)

lambda | TP   | TN   | FP | FN  | precision | recall   | f1_score | accuracy
-------+------+------+----+-----+-----------+----------+----------+---------
2.0    | 7375 | 3797 | 73 | 90  | 0.990199  | 0.987944 | 0.98907  | 0.98562 
1.0    | 7351 | 3816 | 54 | 114 | 0.992708  | 0.984729 | 0.988702 | 0.985179
0.5    | 7292 | 3837 | 33 | 173 | 0.995495  | 0.976825 | 0.986072 | 0.981826
0.1    | 6973 | 3864 | 6  | 492 | 0.99914   | 0.934092 | 0.965522 | 0.956065
0.005  | 6652 | 3869 | 1  | 813 | 0.99985   | 0.891092 | 0.942343 | 0.928187

Saved: ../results/lambda_metrics.csv


In [19]:
# Check which lambda performed best for each metric
best_precision = max(results, key=lambda result: result["precision"])
best_recall = max(results, key=lambda result: result["recall"])
best_f1 = max(results, key=lambda result: result["f1_score"])
best_accuracy = max(results, key=lambda result: result["accuracy"])

print("Best precision lambda:", best_precision["lambda"])
print("Best recall lambda:", best_recall["lambda"])
print("Best F1 lambda:", best_f1["lambda"])
print("Best accuracy lambda:", best_accuracy["lambda"])

# I use F1 as the main choice when precision and recall do not agree
best_lambda = best_f1["lambda"]
best_model = train_smoothed(train_stats, vocabs, best_lambda)

print()
print("Lambda used from here on:", best_lambda)
print()
classify_unknown(unknown_message, best_model)

Best precision lambda: 0.005
Best recall lambda: 2.0
Best F1 lambda: 2.0
Best accuracy lambda: 2.0

Lambda used from here on: 2.0

Prediction: spam
Spam score: -49.04022405065762
Ham score: -59.77816534574214
P(spam | message): 0.9999782948934547
Words used: ['claim', 'congratulations', 'free', 'money', 'now', 'prize', 'won', 'you', 'your']


# 2.3 Improving the Classifier

## Mutual Information Words
For the 200-word version, I ranked words using mutual information between the word appearing in an email and the email class. I used the best lambda from the previous part and skipped words that appeared in fewer than 3 training emails.

In [20]:
# X is whether the word appears, and the class is spam or ham.
# Smoothing keeps the probabilities away from 0.
def mutual_information(p_spam, p_ham):
    p_word = p_spam * prior_spam + p_ham * prior_ham

    mi_score = 0

    for p_class, prior in ((p_spam, prior_spam), (p_ham, prior_ham)):
        # word present
        mi_score += prior * p_class * math.log(p_class / p_word)
        # word absent
        mi_score += prior * (1 - p_class) * math.log((1 - p_class) / (1 - p_word))

    return mi_score

In [21]:
min_docs = 3
top_word_count = 200

def top_informative_words(stats, vocab, lambda_value, skip_frequent=0):
    spam_df, ham_df, spam_doc, ham_doc = stats

    # Skip rare words, then sort so frequent words can be removed if needed
    words = [word for word in vocab if spam_df[word] + ham_df[word] >= min_docs]
    words.sort(key=lambda word: (-(spam_df[word] + ham_df[word]), word))

    word_scores = []

    for word in words[skip_frequent:]:
        p_spam = smoothed_likelihood(spam_df[word], spam_doc, len(vocab), lambda_value)
        p_ham = smoothed_likelihood(ham_df[word], ham_doc, len(vocab), lambda_value)

        word_scores.append({
            "word": word,
            "mi_score": mutual_information(p_spam, p_ham),
            "direction": "spam" if p_spam > p_ham else "ham",
            "spam_df": spam_df[word],
            "ham_df": ham_df[word],
        })

    # Break ties by word so the ranking stays the same each run
    word_scores.sort(key=lambda item: (-item["mi_score"], item["word"]))

    return word_scores[:top_word_count]

In [22]:
# Print the selected words by which class they point to
import textwrap

def print_informative_words(top_words):
    for direction in ("spam", "ham"):
        words = [item["word"] for item in top_words if item["direction"] == direction]

        print(f"Informative words for {direction} ({len(words)}):")
        print(textwrap.fill(", ".join(words), width=100))
        print()

mi_top_200 = top_informative_words(train_stats, vocabs, best_lambda)
mi_top_200_vocabs = {item["word"] for item in mi_top_200}

print("Lambda:", best_lambda)
print()
print_informative_words(mi_top_200)

save_csv(results_dir / "top_200_informative_words.csv", mi_top_200)

Lambda: 2.0

Informative words for spam (127):
mime, format, our, microsoft, outlook, website, express, apache, opt, with, id, from, by, mimeole,
info, message, for, public, produced, sender, esmtp, allowed, v, t, z, effects, company, product,
devmail, sat, l, normal, u, n, sun, news, o, r, ra, stock, foxmail, s, zproxy, b, m, e, your,
writely, promotional, this, p, offer, approved, discreet, aleve, hoodia, ephedra, media, fast, med,
satisfaction, uid, we, ezmlm, hungry, shareholder, day, body, market, pleased, in, exploration,
visit, html, g, campaign, prices, diet, photoshop, legal, us, business, organic, watch, verdana,
acrobat, china, gold, lose, known, premiere, float, professional, corp, investors, announces,
weight, soundly, img, invoked, maximize, retail, bestsellers, wrinkles, awake, revitalizes, zipping,
encore, audition, fatigue, oil, muscle, hair, announce, eat, td, million, week, dvd, lean, today,
life, is, office, ventures, side, world

Informative words for ham (73):
lis

## Testing the 200 Words
Here I train the classifier again using only the selected 200 words, then test it on the same test emails.

In [23]:
# Evaluate using only the smaller vocabulary
def evaluate_vocab(approach, stats, docs, vocab, lambda_value):
    model = train_smoothed(stats, vocab, lambda_value)

    return {
        "approach": approach,
        "vocabulary_size": len(vocab),
        "lambda": lambda_value,
        **evaluate(docs, model),
    }

mi_selected_results = evaluate_vocab("MI 200 words", train_stats, test_docs, mi_top_200_vocabs, best_lambda)

for metric, value in mi_selected_results.items():
    print(metric, ":", value)

approach : MI 200 words
vocabulary_size : 200
lambda : 2.0
TP : 7059
TN : 3650
FP : 220
FN : 406
precision : 0.9697760681412282
recall : 0.9456128600133958
f1_score : 0.9575420510037982
accuracy : 0.944772827525364


## Trying Without the Most Frequent Words
I also tried removing the 200 most frequent words before ranking by mutual information. This checks if common words are hurting the selection.

In [24]:
hovold_top_200 = top_informative_words(train_stats, vocabs, best_lambda, skip_frequent=200)
hovold_vocabs = {item["word"] for item in hovold_top_200}

print_informative_words(hovold_top_200)

hovold_results = evaluate_vocab(
    "MI 200 words, 200 most frequent removed", train_stats, test_docs, hovold_vocabs, best_lambda
)

comparison_columns = ["approach", "vocabulary_size"] + metric_columns

print("Words in both selections:", len(mi_top_200_vocabs & hovold_vocabs))
print()
print_table([mi_selected_results, hovold_results], comparison_columns)
print()
save_csv(results_dir / "mi_attribute_selection_metrics.csv", [mi_selected_results, hovold_results])

Informative words for spam (112):
z, devmail, ra, stock, foxmail, zproxy, writely, promotional, offer, approved, discreet, hoodia,
ephedra, media, fast, med, satisfaction, ezmlm, hungry, shareholder, market, pleased, exploration,
prices, diet, photoshop, legal, business, organic, watch, verdana, acrobat, china, gold, lose,
known, premiere, float, professional, corp, investors, announces, weight, soundly, img, maximize,
retail, bestsellers, wrinkles, awake, revitalizes, zipping, encore, audition, fatigue, oil, muscle,
hair, announce, eat, td, million, week, dvd, lean, today, life, ventures, side, world, money, mood,
nail, shares, aspirin, caffeine, gym, pounds, shipping, mental, stress, solid, headquartered, loss,
optimally, herb, cash, potential, sleep, edition, swear, days, adobe, directors, medium, adverse,
welcomemail, millions, safely, humans, investment, sector, inc, reviews, proprietary, enhancing,
literally, natural, said, drilling, leave, energy

Informative words for ham (88):

# Extra: Cleaned Email Bodies

The required part above reads the whole email file. I added this extra part to see what happens if I only read the email body and remove some obvious noise like HTML words and very short tokens.

## Reading Cleaner Email Text
Here I parse the email body, strip simple HTML, and filter out stopwords and unusual token lengths.

In [25]:
# Email body reader and cleaner
from email import policy
from email.parser import BytesParser
from html import unescape


def strip_html(html_text):
    html_text = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", html_text)
    html_text = re.sub(r"(?s)<[^>]+>", " ", html_text)
    return unescape(html_text)


def read_email_body(path):
    with path.open("rb") as f:
        message = BytesParser(policy=policy.default).parse(f)

    plain_parts = []
    html_parts = []

    if message.is_multipart():
        parts = message.walk()
    else:
        parts = [message]

    for part in parts:
        # Some email parts are malformed, so I just skip those
        try:
            content_type = part.get_content_type()

            if content_type not in {"text/plain", "text/html"}:
                continue

            content = part.get_content()
        except Exception:
            continue

        if not isinstance(content, str):
            continue

        if content_type == "text/plain":
            plain_parts.append(content)
        elif content_type == "text/html":
            html_parts.append(strip_html(content))

    if plain_parts:
        return "\n".join(plain_parts)

    if html_parts:
        return "\n".join(html_parts)

    return ""


stopwords = {
    "the", "and", "for", "you", "your", "that", "this", "with", "from",
    "have", "are", "was", "were", "will", "would", "can", "could",
    "but", "not", "all", "any", "our", "out", "about", "there",
    "their", "they", "them", "then", "than", "has", "had", "his",
    "her", "she", "him", "its", "into", "www", "http", "https",
    "com", "net", "org", "nbsp", "html", "body", "font", "href"
}


def clean_email_words(path):
    # Same word rule as before, with length and stopword filters added
    words = get_words(read_email_body(path))

    return {
        word for word in words
        if 3 <= len(word) <= 20 and word not in stopwords
    }

In [26]:
# Rebuild the counts using the cleaned email bodies
clean_stats = count_documents(train_data, clean_email_words)
clean_spam_df, clean_ham_df, clean_spam_doc, clean_ham_doc = clean_stats

clean_unfiltered_vocabs = set(clean_spam_df) | set(clean_ham_df)

clean_vocabs = {
    word for word in clean_unfiltered_vocabs
    if clean_spam_df[word] + clean_ham_df[word] >= min_docs
}

clean_test_docs = [
    (record["label"], clean_email_words(record["path"]))
    for record in test_data
]

print("Clean unfiltered vocabulary size:", len(clean_unfiltered_vocabs))
print("Clean filtered vocabulary size:", len(clean_vocabs))
print("Empty test bodies:", sum(1 for actual, words in clean_test_docs if not words))

Clean unfiltered vocabulary size: 67836
Clean filtered vocabulary size: 28084
Empty test bodies: 2555


## Lambda on the Cleaned Text
I repeated the lambda test using the cleaned vocabulary.

In [27]:
# Try the same lambdas on the cleaned vocabulary
clean_lambda_results = []

for lambda_value in lambda_values:
    model = train_smoothed(clean_stats, clean_vocabs, lambda_value)

    clean_lambda_results.append({"lambda": lambda_value, **evaluate(clean_test_docs, model)})

print("Improved version lambda results:")
print_table(clean_lambda_results, metric_columns)

best_clean_f1 = max(clean_lambda_results, key=lambda result: result["f1_score"])
best_clean_lambda = best_clean_f1["lambda"]

print()
print("Best precision lambda:", max(clean_lambda_results, key=lambda result: result["precision"])["lambda"])
print("Best recall lambda:", max(clean_lambda_results, key=lambda result: result["recall"])["lambda"])
print("Best F1 lambda:", best_clean_lambda)
print()
save_csv(results_dir / "lambda_metrics_after_cleanup.csv", clean_lambda_results)

Improved version lambda results:
lambda | TP   | TN   | FP  | FN   | precision | recall   | f1_score | accuracy
-------+------+------+-----+------+-----------+----------+----------+---------
2.0    | 6685 | 3767 | 103 | 780  | 0.984826  | 0.895512 | 0.938048 | 0.9221  
1.0    | 6602 | 3770 | 100 | 863  | 0.985079  | 0.884394 | 0.932025 | 0.915042
0.5    | 6458 | 3772 | 98  | 1007 | 0.985052  | 0.865104 | 0.92119  | 0.902514
0.1    | 6204 | 3774 | 96  | 1261 | 0.984762  | 0.831078 | 0.901417 | 0.880282
0.005  | 6299 | 3775 | 95  | 1166 | 0.985142  | 0.843804 | 0.909012 | 0.888752

Best precision lambda: 0.005
Best recall lambda: 2.0
Best F1 lambda: 2.0

Saved: ../results/lambda_metrics_after_cleanup.csv


## Another Way to Pick 200 Words
For this part I picked words based on how different their spam and ham document rates were. I kept 100 spam-leaning words and 100 ham-leaning words.

In [28]:
# Pick informative words using document frequency rates
clean_spam_informative = []
clean_ham_informative = []

for word in clean_vocabs:
    spam_rate = (clean_spam_df[word] + 1) / (clean_spam_doc + 2)
    ham_rate = (clean_ham_df[word] + 1) / (clean_ham_doc + 2)

    rate_score = abs(math.log(spam_rate / ham_rate))

    if spam_rate > ham_rate:
        clean_spam_informative.append((rate_score, word))
    else:
        clean_ham_informative.append((rate_score, word))

clean_spam_informative.sort(key=lambda item: (-item[0], item[1]))
clean_ham_informative.sort(key=lambda item: (-item[0], item[1]))

# Make a 200-word vocabulary: 100 spam words and 100 ham words
small_spam_words = [word for rate_score, word in clean_spam_informative[:100]]
small_ham_words = [word for rate_score, word in clean_ham_informative[:100]]
small_clean_vocabs = set(small_spam_words + small_ham_words)

print("Top spam-informative words:")
print(textwrap.fill(", ".join(small_spam_words), width=100))
print()
print("Top ham-informative words:")
print(textwrap.fill(", ".join(small_ham_words), width=100))
print()

small_docfreq_result = evaluate_vocab(
    "Cleaned document-frequency 200 words", clean_stats, clean_test_docs, small_clean_vocabs, best_clean_lambda
)

for metric, value in small_docfreq_result.items():
    print(metric, ":", value)

Top spam-informative words:
bestsellers, shareholder, bilbo, diversified, penis, cantex, ctxe, maurin, bbb, infinex, inifinex,
texada, audition, rolex, ters, helors, clisk, heuer, minfile, skarns, yew, bvlgari, breitling,
patek, audemars, panerai, piguet, vacheron, homeowner, seee, niobrara, encore, shareholders, gapj,
ventures, torate, expec, guaranted, erections, sperm, daytona, iwc, officine, sohne, meds,
campaings, redited, thorin, xed, mineral, tenacious, profitability, ixed, remitted, cialis, rees,
tigious, cartier, ities, weeek, cgdc, unexplored, stockholders, coatings, ject, swath, premiere,
baggins, retail, booming, lomas, orgasms, dwarves, geopolitical, golfing, sifted, superstar, hina,
hoodia, hobbit, ower, clixme, cried, georgiadis, greko, passport, gol, goblins, mas, bac, redi,
herbal, winnings, imageready, wihtout, headquartered, pills, bellwetherreport, climbs, cwtd

Top ham-informative words:
dmdx, ascii, ribbon, handyboard, handy, robot, motors, sensor, interval, nodes

## Mutual Information on Cleaned Text
I also ran the mutual information selection again using the cleaned words.

In [29]:
clean_mi_top_200 = top_informative_words(clean_stats, clean_vocabs, best_clean_lambda)
clean_hovold_top_200 = top_informative_words(clean_stats, clean_vocabs, best_clean_lambda, skip_frequent=200)

print_informative_words(clean_mi_top_200)

clean_mi_result = evaluate_vocab(
    "Cleaned MI 200 words", clean_stats, clean_test_docs,
    {item["word"] for item in clean_mi_top_200}, best_clean_lambda
)

clean_hovold_result = evaluate_vocab(
    "Cleaned MI 200 words, 200 most frequent removed", clean_stats, clean_test_docs,
    {item["word"] for item in clean_hovold_top_200}, best_clean_lambda
)

print_table([clean_mi_result, clean_hovold_result], comparison_columns)

Informative words for spam (58):
company, approved, offer, shareholder, pleased, market, exploration, retail, photoshop, gold,
acrobat, inc, premiere, oil, announces, corp, business, bestsellers, shares, investors,
professional, stock, audition, encore, announce, prices, million, adobe, dvd, news, money, ventures,
potential, today, headquartered, watch, pro, price, sector, edition, directors, trading, shipping,
cash, gas, investment, reviews, natural, enhancing, drilling, producer, days, risk, companies,
reserves, marketing, effects, technologies

Informative words for ham (142):
using, use, thanks, problem, university, does, dmdx, anyone, ascii, ribbon, list, handyboard, handy,
work, running, file, how, against, message, files, however, seems, port, mail, input, think, code,
output, know, same, robot, control, set, run, display, systems, version, probably, two, item, other,
test, either, also, fred, between, tried, machine, motor, error, send, problems, computer, uses,
engineering, di

## Comparing the Runs
This table puts the main runs beside each other. The first rows are the required experiments, and the cleaned-body rows are extra.

In [30]:
# Compare the main approaches I tried
best_clean_result = max(clean_lambda_results, key=lambda result: result["f1_score"])

comparison_results = [
    {"approach": "Full vocabulary, no smoothing", "vocabulary_size": len(vocabs), "lambda": "none", **baseline_result},
    {"approach": "Full vocabulary, best lambda", "vocabulary_size": len(vocabs), **best_f1},
    mi_selected_results,
    hovold_results,
    {"approach": "Cleaned full vocabulary", "vocabulary_size": len(clean_vocabs), **best_clean_result},
    small_docfreq_result,
    clean_mi_result,
    clean_hovold_result,
]

print("Overall evaluation comparison:")
print_table(comparison_results, comparison_columns)

best_overall = max(comparison_results, key=lambda result: result["f1_score"])

print()
print("Best approach by F1 score:", best_overall["approach"])
print("Best F1 score:", round(best_overall["f1_score"], 6))
print("Best precision:", round(best_overall["precision"], 6))
print("Best recall:", round(best_overall["recall"], 6))
print("Best accuracy:", round(best_overall["accuracy"], 6))

Overall evaluation comparison:
approach                                        | vocabulary_size | lambda | TP   | TN   | FP   | FN   | precision | recall   | f1_score | accuracy
------------------------------------------------+-----------------+--------+------+------+------+------+-----------+----------+----------+---------
Full vocabulary, no smoothing                   | 78278           | none   | 6851 | 3866 | 4    | 614  | 0.999416  | 0.917749 | 0.956844 | 0.945479
Full vocabulary, best lambda                    | 78278           | 2.0    | 7375 | 3797 | 73   | 90   | 0.990199  | 0.987944 | 0.98907  | 0.98562 
MI 200 words                                    | 200             | 2.0    | 7059 | 3650 | 220  | 406  | 0.969776  | 0.945613 | 0.957542 | 0.944773
MI 200 words, 200 most frequent removed         | 200             | 2.0    | 7074 | 3661 | 209  | 391  | 0.971303  | 0.947622 | 0.959317 | 0.947067
Cleaned full vocabulary                         | 28084           | 2.0    | 6685

In [31]:
# Save the comparison table for the report
save_csv(results_dir / "overall_approach_comparison.csv", comparison_results)

comparison_md = results_dir / "overall_approach_comparison.md"

with comparison_md.open("w") as f:
    f.write("# Overall Approach Comparison\n\n")
    f.write("| Approach | Vocab | Lambda | TP | TN | FP | FN | Precision | Recall | F1 | Accuracy |\n")
    f.write("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|\n")

    for result in comparison_results:
        f.write(
            f"| {result['approach']} | {result['vocabulary_size']} | {result['lambda']} | "
            f"{result['TP']} | {result['TN']} | {result['FP']} | {result['FN']} | "
            f"{result['precision']:.6f} | {result['recall']:.6f} | "
            f"{result['f1_score']:.6f} | {result['accuracy']:.6f} |\n"
        )

    f.write("\n")
    f.write(f"Best approach by F1 score: `{best_overall['approach']}`.\n")

print("Saved:", comparison_md)

Saved: ../results/overall_approach_comparison.csv
Saved: ../results/overall_approach_comparison.md
